In [0]:
# Databricks notebook source

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_TABLE = "marathos.bronze.races_raw"
SILVER_TABLE = "marathos.silver.races_obt"

MIN_EVENT_YEAR = 2010
MIN_SPEED_KMH = 2.0
MAX_SPEED_KMH = 25.0

# COMMAND ----------

bronze = spark.table(BRONZE_TABLE)

display(bronze.limit(10))

# COMMAND ----------

df = bronze.withColumnRenamed("athlete_id", "athlete_id_source")

# COMMAND ----------

df = (
    df
    .withColumn("event_year", F.col("year_of_event").cast("int"))
    .withColumn(
        "event_date_text",
        F.regexp_extract(F.col("event_dates"), r"(\d{2}\.\d{2}\.\d{4})", 1)
    )
    .withColumn(
        "event_date",
        F.to_date(F.col("event_date_text"), "dd.MM.yyyy")
    )
    .withColumn("event_name", F.trim(F.col("event_name")))
    .withColumn("event_distance_raw", F.trim(F.col("event_distance_length")))
    .withColumn("performance_raw", F.trim(F.col("athlete_performance")))
    .withColumn("athlete_country", F.trim(F.col("athlete_country")))
    .withColumn("athlete_gender", F.trim(F.col("athlete_gender")))
    .withColumn("athlete_birth_year", F.col("athlete_year_of_birth").cast("int"))
    .withColumn("event_finishers", F.col("event_number_of_finishers").cast("int"))
)

# COMMAND ----------

df = (
    df
    .withColumn(
        "distance_value_text",
        F.regexp_extract(F.col("event_distance_raw"), r"^([\d.]+)", 1)
    )
    .withColumn(
        "distance_value",
        F.expr("try_cast(distance_value_text as double)")
    )
    .withColumn(
        "distance_unit",
        F.lower(F.regexp_extract(F.col("event_distance_raw"), r"([a-zA-Z]+)$", 1))
    )
)

# COMMAND ----------

df = (
    df
    .withColumn(
        "performance_unit",
        F.lower(F.regexp_extract(F.col("performance_raw"), r"([a-zA-Z]+)$", 1))
    )
    .withColumn(
        "performance_time",
        F.regexp_replace(F.lower(F.col("performance_raw")), r"\s*h$", "")
    )
)

# COMMAND ----------

valid_start = (
    df
    .filter(F.col("event_year") >= MIN_EVENT_YEAR)
    .filter(F.col("event_name").isNotNull())
    .filter(F.col("performance_raw").isNotNull())
    .filter(F.col("distance_unit").isin("km", "mi"))
    .filter(~F.lower(F.col("performance_raw")).contains("d"))
    .filter(~F.lower(F.col("event_name")).contains("etappen"))
    .filter(~F.lower(F.col("event_name")).contains("etap"))
)

# COMMAND ----------

valid_start = valid_start.withColumn("time_parts", F.split(F.col("performance_time"), ":"))

# COMMAND ----------

valid_start = (
    valid_start
    .withColumn("time_part_1", F.expr("try_cast(time_parts[0] as double)"))
    .withColumn("time_part_2", F.expr("try_cast(time_parts[1] as double)"))
    .withColumn("time_part_3", F.expr("try_cast(time_parts[2] as double)"))
)

# COMMAND ----------

df_time = (
    valid_start
    .withColumn(
        "performance_seconds",
        F.when(
            F.size(F.col("time_parts")) == 3,
            F.col("time_part_1") * 3600 + F.col("time_part_2") * 60 + F.col("time_part_3")
        )
        .when(
            F.size(F.col("time_parts")) == 2,
            F.col("time_part_1") * 60 + F.col("time_part_2")
        )
    )
    .drop("time_parts", "time_part_1", "time_part_2", "time_part_3")
)

# COMMAND ----------

df_distance = (
    df_time
    .withColumn(
        "distance_km",
        F.when(F.col("distance_unit") == "km", F.col("distance_value"))
        .when(F.col("distance_unit") == "mi", F.col("distance_value") * 1.60934)
    )
)

# COMMAND ----------

df_speed = (
    df_distance
    .withColumn(
        "speed_text",
        F.regexp_replace(F.col("athlete_average_speed"), ",", ".")
    )
    .withColumn(
        "speed_kmh_raw",
        F.expr("try_cast(speed_text as double)")
    )
    .withColumn(
        "speed_kmh_calculated",
        F.when(
            (F.col("distance_km").isNotNull()) & (F.col("performance_seconds") > 0),
            F.col("distance_km") / (F.col("performance_seconds") / 3600)
        )
    )
    .withColumn(
        "speed_kmh",
        F.when(
            F.col("speed_kmh_raw").between(MIN_SPEED_KMH, MAX_SPEED_KMH),
            F.col("speed_kmh_raw")
        )
        .when(
            F.col("speed_kmh_calculated").between(MIN_SPEED_KMH, MAX_SPEED_KMH),
            F.col("speed_kmh_calculated")
        )
    )
    .drop("speed_text")
)

# COMMAND ----------

df_age = df_speed.withColumn(
    "athlete_age",
    F.col("event_year") - F.col("athlete_birth_year")
)

# COMMAND ----------

valid = (
    df_age
    .filter(F.col("distance_km").isNotNull())
    .filter(F.col("performance_seconds").isNotNull())
    .filter(F.col("speed_kmh").isNotNull())
    .filter(F.col("athlete_age").between(10, 90))
)

# COMMAND ----------

marathon_scope = (
    valid
    .filter(
        (F.col("distance_km").between(41.0, 43.5))
        | (F.lower(F.col("event_name")).contains("marathon"))
    )
)

# COMMAND ----------

event_window = Window.orderBy("event_name")

athlete_window = Window.orderBy(
    "athlete_country",
    "athlete_birth_year",
    "athlete_gender",
    "athlete_id_source"
)

result_window = Window.orderBy(
    "event_year",
    "event_name",
    "athlete_id_source",
    "performance_seconds"
)

obt = (
    marathon_scope
    .withColumn("event_id", F.dense_rank().over(event_window))
    .withColumn("athlete_id", F.dense_rank().over(athlete_window))
    .withColumn("result_id", F.row_number().over(result_window))
)

# COMMAND ----------

obt = (
    obt
    .select(
        "result_id",
        "event_id",
        "athlete_id",
        "event_year",
        "event_date",
        "event_name",
        "event_distance_raw",
        "distance_km",
        "distance_unit",
        "event_finishers",
        "performance_raw",
        "performance_seconds",
        "athlete_club",
        "athlete_country",
        "athlete_birth_year",
        "athlete_gender",
        "athlete_age_category",
        "athlete_age",
        "speed_kmh",
        "athlete_id_source"
    )
)

# COMMAND ----------

print("Bronze rows:", bronze.count())
print("Silver rows:", obt.count())

display(obt.limit(20))

# COMMAND ----------

(
    obt.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

# COMMAND ----------

display(
    spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {SILVER_TABLE}
    """)
)

display(spark.table(SILVER_TABLE).limit(20))